In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from transformers import RobertaModel #Model used
from transformers import RobertaTokenizer
from transformers import AdamW
from sklearn.metrics import f1_score

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Specify the dataset path (adjust the path if needed)
dataset_path = '/content/drive/My Drive/SemEval'

In [ ]:
# Load the CSV file
def load_data(file_path,delimeter = ","):
    df = pd.read_csv(file_path, delimiter = delimeter)
    texts = df["text"].tolist()
    labels = df[["anger", "fear", "joy", "sadness", "surprise"]].values
    return texts, labels

# Load train and test data
train_file = "/content/drive/My Drive/SemEval/track_a/train/eng.csv"
test_file = "/content/drive/My Drive/SemEval/track_a/dev/eng.csv"
human_pred = "/content/drive/My Drive/SemEval/track_a/Human_eval_eng.csv"

train_texts, train_labels = load_data(train_file)
test_texts, test_labels = load_data(test_file)
h_texts, h_labels = load_data(human_pred)



# Initialize tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

# Find the max token length dynamically
def find_max_length(texts, tokenizer):
    tokenized_texts = [tokenizer.tokenize(text) for text in texts]
    return max(len(tokens) for tokens in tokenized_texts)

max_length = find_max_length(train_texts + test_texts, tokenizer)  # Find max length from both train & test

print(f"Dynamic max length: {max_length}")

# Tokenize with dynamic max_length
def tokenize_texts(texts, tokenizer, max_length):
    return tokenizer(
        texts,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

train_encodings = tokenize_texts(train_texts, tokenizer, max_length)
test_encodings = tokenize_texts(test_texts, tokenizer, max_length)
h_encodings = tokenize_texts(h_texts, tokenizer, max_length)



EmotionDataset: Custom PyTorch Dataset for Emotion Classification.

- Stores tokenized inputs (`input_ids`, `attention_mask`) and labels (if provided).
- Converts inputs and labels to PyTorch tensors and moves them to the specified device (`cpu` or `cuda`).
- Implements `__len__()` to return the dataset size.
- Implements `__getitem__()` to retrieve a sample in dictionary format.
- Supports labeled data (for training) and unlabeled data (for inference).

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels=None, device="cpu"):
        """
        Custom PyTorch Dataset for Emotion Classification.

        Args:
            encodings (dict): Tokenized input (should contain "input_ids" and "attention_mask").
            labels (list or tensor, optional): Corresponding emotion labels (multi-label format).
            device (str): Device to store tensors ('cpu' or 'cuda').
        """
        self.encodings = {key: torch.tensor(val, dtype=torch.long).to(device) for key, val in encodings.items()}
        self.labels = torch.tensor(labels, dtype=torch.float).to(device) if labels is not None else None

    def __len__(self):
        """ Returns the total number of samples in the dataset. """
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        """ Returns one sample (dictionary format). """
        item = {key: val[idx] for key, val in self.encodings.items()}  # Get input tensors
        if self.labels is not None:
            item["labels"] = self.labels[idx]  # Add labels only if available
        return item

# Example Usage
train_dataset = EmotionDataset(train_encodings, train_labels, device="cuda")
test_dataset = EmotionDataset(test_encodings, test_labels, device="cuda")
h_dataset = EmotionDataset(h_encodings,test_labels, device="cuda")  # No labels (for inference)



RobertaClass: A text classification model based on RoBERTa.

- Uses a pre-trained "roberta-large" model to extract contextual embeddings.
- The [CLS] token representation (pooler_output) is used for classification.
- Three fully connected layers refine the representation:
  - fc1 (1024 → 512) with LayerNorm, GELU activation, and Dropout.
  - fc2 (512 → 256) with LayerNorm, GELU activation, and Dropout.
  - fc3 (256 → num_labels) for final classification logits.


- LayerNorm (Layer Normalization): Normalizes activations across features, stabilizing training and improving convergence.
- Relu is used
- Dropout: Randomly drops neurons during training to prevent overfitting and improve generalization.


In [ ]:
import torch
import torch.nn as nn
import random
import numpy as np
from transformers import RobertaModel

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

class RobertaClass(nn.Module):
    def __init__(self, num_labels=5):
        super(RobertaClass, self).__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-large")
        self.dropout = nn.Dropout(0.1)  # Reduced dropout to 0.1

        # Fully connected layers with RELU activation and LayerNorm
        self.fc1 = nn.Linear(1024, 512)
        self.layer_norm1 = nn.LayerNorm(512)
        self.fc2 = nn.Linear(512, 256)
        self.layer_norm2 = nn.LayerNorm(256)
        self.fc3 = nn.Linear(256, num_labels)

        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        output = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = output.pooler_output  # Use pooler output for better classification

        x = self.fc1(cls_output)
        x = self.layer_norm1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.layer_norm2(x)
        x = self.relu(x)
        x = self.dropout(x)

        logits = self.fc3(x)

        return logits


In [ ]:
# Initialize model
num_labels = 5  # Since we are predicting 5 emotions
model = RobertaClass(num_labels=num_labels)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


DataLoaders for batching and shuffling:

- `BATCH_SIZE = 16`: Defines how many samples per batch.
- `train_loader`: Loads training data with shuffling (`shuffle=True`) to improve generalization.
- `test_loader`: Loads test data without shuffling (`shuffle=False`) to ensure consistent evaluation.
- `DataLoader` handles batching, shuffling, and efficient data loading for training and inference.

In [ ]:
# Define batch size
BATCH_SIZE = 16

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)



Loss Function & Optimizer:

- `BCEWithLogitsLoss()`: Used for multi-label classification. Combines sigmoid activation with binary cross-entropy loss.
- `AdamW`: Optimizer designed for transformers.
  - `lr=1e-5`: Learning rate (controls step size during optimization).
  - `weight_decay=1e-2`: Regularization to prevent overfitting.

In [ ]:
# Define loss function for multi-label classification
criterion = nn.BCEWithLogitsLoss()

# Define optimizer
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
#optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

Training Loop & Learning Rate Scheduler:

- `device`: Moves model to GPU (if available) for faster training.
- `get_scheduler("linear")`: Implements a linear learning rate decay.
- `epochs = 20`: Trains for 20 iterations over the dataset.
- `num_training_steps`: Total steps based on `epochs * dataset size`.
- `tqdm`: Displays progress bar for each training batch.

Training Process:
1. Set model to `train()` mode.
2. Iterate through `train_loader`, moving data to the device.
3. Compute predictions and calculate loss (`BCEWithLogitsLoss`).
4. Perform backpropagation (`loss.backward()`).
5. Update model weights (`optimizer.step()`).
6. Adjust learning rate (`lr_scheduler.step()`).
7. Print progress and total loss per epoch.

In [ ]:
import torch
from transformers import get_scheduler
from tqdm import tqdm
import matplotlib.pyplot as plt

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Learning rate scheduler
epochs =50
num_training_steps = epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# List to record average loss per epoch
loss_history = []

# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f}")

# Plot the training loss over epochs
plt.figure(figsize=(8, 6))
plt.plot(range(1, epochs + 1), loss_history, marker='o', linestyle='-', color='b')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('Training Loss per Epoch')
plt.grid(True)
plt.show()



Evaluation Function:

- `model.eval()`: Sets the model to evaluation mode (disables dropout & gradient updates).
- `torch.no_grad()`: Disables gradient computation for faster inference.
- Iterates over `test_loader`, moving data to the correct device.
- Computes model outputs and applies `sigmoid()` to get probabilities.
- Converts probabilities to binary predictions using a threshold of 0.5.
- Uses `classification_report()` to generate precision, recall, and F1-score.



In [ ]:
from sklearn.metrics import classification_report
import numpy as np

def evaluate(model, test_loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.sigmoid(outputs).cpu().numpy()  # Convert logits to probabilities
            labels = labels.cpu().numpy()

            predictions.extend(preds)
            true_labels.extend(labels)

    # Convert probabilities to binary values (threshold = 0.5)
    predictions = np.array(predictions) > 0.5

    print(classification_report(true_labels, predictions, target_names=["anger", "fear", "joy", "sadness", "surprise"]))

# Evaluate the model
evaluate(model, test_loader)


Relu

In [ ]:
model_save_path = "/content/drive/My Drive/SemEval/roberta_emotion_model_Relu.pt"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
# Evaluation with human predictions
model.eval()
all_preds = []
all_labels = []
test_loss = 0.0
correct = 0
total = 0

h_loader = DataLoader(h_dataset, batch_size=8, shuffle=False)

with torch.no_grad():
    for batch in h_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask)

        # Apply sigmoid and threshold to get binary predictions
        preds = (torch.sigmoid(outputs) > 0.5).float()  # Threshold at 0.5 for multi-label

        # Collect predictions and labels for F1 score
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# Flatten lists and calculate F1 score
all_preds = np.vstack(all_preds)  # Shape: (num_samples, num_labels)
all_labels = np.vstack(all_labels)  # Shape: (num_samples, num_labels)

# Calculate F1 score for each label individually
f1_per_label = f1_score(all_labels, all_preds, average=None)  # F1 score for each label
print("F1 Score per label:")
for idx, score in enumerate(f1_per_label):
    print(f"Label {idx} F1 Score: {score:.4f}")



In [ ]:
# Load model
model.load_state_dict(torch.load("/content/drive/My Drive/SemEval/roberta_emotion_model_Relu.pt", map_location=device))
model.eval()

# Function to predict emotions
def predict_emotions(text, model, tokenizer):
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        probs = torch.sigmoid(output).cpu().numpy()[0]  # Convert logits to probabilities

    emotions = ["anger", "fear", "joy", "sadness", "surprise"]
    result = {emotion: round(float(prob), 2) for emotion, prob in zip(emotions, probs)}

    return result

# Example inference
text = "I cannot wait for the trip!"
print(predict_emotions(text, model, tokenizer))


Unlabeled test data is predicted using the trained model. Now test data has labels

In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import pandas as pd
import numpy as np

# Set device to CUDA:4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Define model and tokenizer
MODEL_PATH = "roberta_emotion_model_Relu.pt"
NUM_LABELS = 5  # Adjust based on your model

model.load_state_dict(torch.load("roberta_emotion_model_Relu.pt", map_location=DEVICE))
model.to(DEVICE)
model.eval()

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Define emotion labels
EMOTION_LABELS = ["anger", "fear", "joy", "sadness", "surprise"]

# Define max sequence length
MAX_LENGTH = 512

# Load test dataset
test_file = "//content/drive/My Drive/SemEval/track_a/test/eng.csv"
df = pd.read_csv(test_file)

# Ensure text column exists
TEXT_COLUMN = "text"
if TEXT_COLUMN not in df.columns:
    raise ValueError(f"Column '{TEXT_COLUMN}' not found in CSV!")

# Function to predict emotions with probability scores
def predict_emotions(text, model, tokenizer):
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")

    # Move tensors to CUDA:2
    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        probs = torch.sigmoid(output).cpu().numpy()[0]  # Convert logits to probabilities

    # Return dictionary of emotion probabilities
    return {emotion: round(float(prob), 2) for emotion, prob in zip(EMOTION_LABELS, probs)}

# Perform inference on the test dataset
predictions = df[TEXT_COLUMN].fillna("").apply(lambda text: predict_emotions(text, model, tokenizer))

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions.tolist())


In [ ]:
# Overwrite the existing label columns
df = df.drop(columns=[col for col in df.columns if col in EMOTION_LABELS], errors="ignore")  # Remove old labels
df = pd.concat([df[[TEXT_COLUMN]], predictions_df], axis=1)  # Keep only text and new labels

# Save results with correct CSV formatting
output_file = "test_predictions_with_probs.csv"
df.to_csv(output_file, index=False, sep=",", encoding="utf-8", na_rep="")

print(f"Predictions saved to {output_file}")

In [ ]:
import torch
import shap
from transformers import RobertaTokenizer
import matplotlib.pyplot as plt

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RobertaClass(num_labels=5)  # Your model class
model.load_state_dict(torch.load("roberta_emotion_model_Relu.pt"), strict=False)  # Load trained weights
model.to(device)
model.eval()

class ModelWrapper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def __call__(self, masked_texts):
        """Accept masked texts from SHAP, tokenize them, and return model outputs."""
        # Ensure input is a list of strings
        if isinstance(masked_texts, str):
            masked_texts = [masked_texts]
        # Tokenize text manually and ensure attention mask is handled correctly
        encoded_inputs = [self.tokenizer(text, padding="max_length", truncation=True, return_tensors="pt") for text in masked_texts]
        input_ids = torch.cat([x["input_ids"] for x in encoded_inputs]).to(device)
        attention_mask = torch.cat([x["attention_mask"] for x in encoded_inputs]).to(device)
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask)
            probs = torch.sigmoid(outputs)  # Convert logits to probabilities
        return probs.cpu().numpy()  # Ensure SHAP receives NumPy array

# Use a SHAP Masker for Token-based Models
masker = shap.maskers.Text(tokenizer, collapse_mask_token="[MASK]")

# Initialize SHAP Explainer
wrapper = ModelWrapper(model, tokenizer)
explainer = shap.Explainer(wrapper, masker, output_names=EMOTION_LABELS)

# Extract and clean sample texts from `test_loader`
sample_size = 3  # Adjust based on how many examples you want to analyze
sample_texts = []

for batch in test_loader:
    input_ids = batch["input_ids"][:sample_size].to(device)  # Get sample input IDs

    # Decode the input IDs to text
    decoded_texts = [
        tokenizer.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        for ids in input_ids
    ]

    sample_texts.extend(decoded_texts)

    if len(sample_texts) >= sample_size:
        break  # Stop after collecting enough examples

# Compute SHAP values with clean inputs
shap_values = explainer(sample_texts)
print(sample_texts)

# Ensure we iterate only over computed SHAP values
for i in range(len(shap_values)):
    print(f"\n🔹 Sample {i+1}: {sample_texts[i]}")
    shap_output = shap_values[i]
    # Clean the SHAP output by removing "Ġ" characters
    shap_output.data = [token.replace("Ġ", " ").strip() for token in shap_output.data]
    # Cleaning SHAP output tokens

    print(f"shap_output_data: {shap_output.data}")
    shap.plots.text(shap_output)


In [ ]:
shap.plots.bar(shap_values[:, :, "joy"].mean(0))

In [ ]:
shap.plots.text(shap_values[:, :, "anger"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data extracted from the table
data = {
    "Experiment": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
    "Model": ["RoBERTa"] * 9 + ["ALBERT"] * 4,
    "Mean F1 Score": [0.778, 0.78, 0.782, 0.75, 0.744, 0.75, 0.736, 0.776, 0.774, 0.72, 0.776, 0.66, 0.684],
    "Variance": [0.001336, 0.000584, 0.001336, 0.0018, 0.000784, 0.000464, 0.03816, 0.001864, 0.001864, 0.000464, 0.001864, 0.005324, 0.007344],
    "Training Loss": [0.0747, 0.0568, 0.0314, 0.2749, 0.1033, 0.0516, 0.0061, 0.0363, 0.0189, 0.0236, 0.0236, 0.0834, 0.0953],
    "F1 Scores": {
        "anger": [0.76, 0.77, 0.74, 0.67, 0.71, 0.73, 0.71, 0.81, 0.81, 0.71, 0.81, 0.62, 0.59],
        "fear": [0.81, 0.82, 0.83, 0.78, 0.76, 0.82, 0.73, 0.81, 0.81, 0.82, 0.81, 0.75, 0.74],
        "joy": [0.73, 0.78, 0.81, 0.75, 0.71, 0.69, 0.77, 0.82, 0.73, 0.77, 0.82, 0.54, 0.57],
        "sadness": [0.76, 0.75, 0.74, 0.75, 0.77, 0.76, 0.77, 0.73, 0.73, 0.76, 0.73, 0.77, 0.75],
        "surprise": [0.83, 0.75, 0.79, 0.71, 0.77, 0.75, 0.46, 0.73, 0.73, 0.76, 0.73, 0.69, 0.77],
    },
}
'''
# 1. Training Loss vs Mean F1 Score with Variance
plt.figure(figsize=(12, 6))
colors = ["blue" if model == "RoBERTa" else "orange" for model in data["Model"]]
sizes = [100 + variance * 10000 for variance in data["Variance"]]  # Bubble size based on variance
plt.scatter(data["Training Loss"], data["Mean F1 Score"], c=colors, s=sizes, alpha=0.8, label="Experiments")
for i, experiment in enumerate(data["Experiment"]):
    plt.text(data["Training Loss"][i], data["Mean F1 Score"][i], f"Exp {experiment}", fontsize=8)
plt.title("Training Loss vs Mean F1 Score with Variance")
plt.xlabel("Training Loss")
plt.ylabel("Mean F1 Score")
plt.grid()
plt.tight_layout()
plt.show()'''

# 2. Per-Label F1 Score Comparison (Line Chart)
plt.figure(figsize=(12, 6))
for label, scores in data["F1 Scores"].items():
    plt.plot(data["Experiment"], scores, label=label, marker="o", linewidth=2)
plt.title("Per-Label F1 Score Comparison")
plt.xlabel("Experiment")
plt.ylabel("F1 Score")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

# Scatter Plot: Training Loss vs Variance
plt.figure(figsize=(12, 6))
colors = ["blue" if model == "RoBERTa" else "orange" for model in data["Model"]]
sizes = [100 + score * 100 for score in data["Mean F1 Score"]]  # Bubble size based on Mean F1 Score
plt.scatter(data["Training Loss"], data["Variance"], c=colors, s=sizes, alpha=0.8)

# Add Annotations for Experiments
for i, experiment in enumerate(data["Experiment"]):
    plt.text(data["Training Loss"][i], data["Variance"][i], f"Exp {experiment}", fontsize=8)

# Titles and Labels
plt.title("Training Loss vs Variance Across Experiments")
plt.xlabel("Training Loss")
plt.ylabel("Variance")
plt.grid()
plt.tight_layout()
plt.show()

# Scatter Plot: Variance vs Mean F1 Score
plt.figure(figsize=(10, 6))
colors = ["blue" if model == "RoBERTa" else "orange" for model in data["Model"]]
sizes = [100 + loss * 100 for loss in data["Training Loss"]]  # Optional: Bubble size based on training loss
plt.scatter(data["Mean F1 Score"], data["Variance"], c=colors, s=sizes, alpha=0.8)

# Add Annotations for Experiments
for i, experiment in enumerate(data["Experiment"]):
    plt.text(data["Mean F1 Score"][i], data["Variance"][i], f"Exp {experiment}", fontsize=8)

# Titles and Labels
plt.title("Variance vs Mean F1 Score Across Experiments")
plt.xlabel("Mean F1 Score")
plt.ylabel("Variance")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data extracted from the table
data = {
    "Experiment": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
    "Model": ["RoBERTa"] * 9 + ["ALBERT"] * 4,
    "Mean F1 Score": [0.778, 0.78, 0.782, 0.75, 0.744, 0.75, 0.736, 0.776, 0.774, 0.72, 0.776, 0.66, 0.684],
    "Variance": [0.001336, 0.000584, 0.001336, 0.0018, 0.000784, 0.000464, 0.03816, 0.001864, 0.001864, 0.000464, 0.001864, 0.005324, 0.007344],
}

# Calculate Standard Deviation (SD) from Variance
data["SD"] = [np.sqrt(var) for var in data["Variance"]]

# Bar Plot: SD vs Mean F1 Score
plt.figure(figsize=(12, 6))
bar_width = 0.4

# Create bars for RoBERTa and ALBERT separately
roberta_indices = [i for i, model in enumerate(data["Model"]) if model == "RoBERTa"]
albert_indices = [i for i, model in enumerate(data["Model"]) if model == "ALBERT"]

roberta_means = [data["Mean F1 Score"][i] for i in roberta_indices]
roberta_sds = [data["SD"][i] for i in roberta_indices]

albert_means = [data["Mean F1 Score"][i] for i in albert_indices]
albert_sds = [data["SD"][i] for i in albert_indices]

# Plot bars for RoBERTa
plt.bar(
    [x - bar_width / 2 for x in range(len(roberta_means))],
    roberta_sds,
    width=bar_width,
    label="RoBERTa SD",
    color="blue",
    alpha=0.8,
)
plt.bar(
    [x - bar_width / 2 for x in range(len(roberta_means))],
    roberta_means,
    width=bar_width,
    bottom=roberta_sds,
    label="RoBERTa Mean F1",
    color="lightblue",
    alpha=0.8,
)

# Plot bars for ALBERT
plt.bar(
    [x + len(roberta_means) + bar_width / 2 for x in range(len(albert_means))],
    albert_sds,
    width=bar_width,
    label="ALBERT SD",
    color="orange",
    alpha=0.8,
)
plt.bar(
    [x + len(roberta_means) + bar_width / 2 for x in range(len(albert_means))],
    albert_means,
    width=bar_width,
    bottom=albert_sds,
    label="ALBERT Mean F1",
    color="peachpuff",
    alpha=0.8,
)

# Titles and Labels
plt.title("Standard Deviation and Mean F1 Score Across Experiments")
plt.xlabel("Experiment Index (RoBERTa and ALBERT)")
plt.ylabel("Scores")
plt.xticks(
    list(range(len(roberta_means))) + [x + len(roberta_means) for x in range(len(albert_means))],
    data["Experiment"],
)
plt.legend()
plt.tight_layout()
plt.show()
